# Задание 3. Рекуррентные сети (LSTM и GRU)

План: подбираем параметры по очереди (один параметр меняем, остальные держим лучшими на данный момент),
оцениваем на **валидации**, а на **тесте** считаем метрики один раз для финальных моделей.

In [ ]:
import os, re, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, classification_report, ConfusionMatrixDisplay

SEED = 42
keras.utils.set_random_seed(SEED)
os.makedirs("../results", exist_ok=True)

# ---- настройки ----
TRAIN_PATH, TEST_PATH = "../data/train.parquet", "../data/test.parquet"
TEXT_COL, LABEL_COL = "CONTENT_edited", "class"
VOCAB = 20000        # размер словаря
EPOCHS = 30          # максимум эпох (есть EarlyStopping)
BATCH = 64
PREPROC = "clean+typos+lemma+slang"     # название предобработки для таблицы результатов
USE_CLASS_WEIGHT = True
print(tf.__version__, tf.config.list_physical_devices("GPU"))

In [ ]:
def clean(text):
    return str(text)

train_df = pd.read_parquet(TRAIN_PATH).dropna(subset=[TEXT_COL, LABEL_COL])
test_df = pd.read_parquet(TEST_PATH).dropna(subset=[TEXT_COL, LABEL_COL])

le = LabelEncoder().fit(train_df[LABEL_COL])
N_CLASSES = len(le.classes_)

X_text = train_df[TEXT_COL].map(clean).values
y = le.transform(train_df[LABEL_COL])
X_test_text = test_df[TEXT_COL].map(clean).values
y_test = le.transform(test_df[LABEL_COL])

# валидация: 15% от train, со стратификацией
X_tr_text, X_val_text, y_tr, y_val = train_test_split(
    X_text, y, test_size=0.15, stratify=y, random_state=SEED)

print("train/val/test:", len(y_tr), len(y_val), len(y_test), "| классов:", N_CLASSES)
print(pd.Series(le.inverse_transform(y_tr)).value_counts())

In [ ]:
# длина последовательности: 95-й перцентиль числа слов (но не больше 300)
lengths = [len(t.split()) for t in X_tr_text]
MAXLEN = int(min(np.percentile(lengths, 95), 300))
print("MAXLEN =", MAXLEN)

vectorizer = layers.TextVectorization(max_tokens=VOCAB, output_sequence_length=MAXLEN)
vectorizer.adapt(X_tr_text)

def vec(texts):
    return vectorizer(np.array(texts, dtype=str)).numpy()

X_tr, X_val, X_test = vec(X_tr_text), vec(X_val_text), vec(X_test_text)
Y_tr = keras.utils.to_categorical(y_tr, N_CLASSES)
Y_val = keras.utils.to_categorical(y_val, N_CLASSES)

if USE_CLASS_WEIGHT:
    w = compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=y_tr)
    CLASS_WEIGHT = dict(enumerate(w))
else:
    CLASS_WEIGHT = None

In [ ]:
OPTIMIZERS = {
    "adam":    lambda: keras.optimizers.Adam(1e-3),
    "nadam":   lambda: keras.optimizers.Nadam(1e-3),
    "rmsprop": lambda: keras.optimizers.RMSprop(1e-3),
    "adamw":   lambda: keras.optimizers.AdamW(1e-3),
    "sgd":     lambda: keras.optimizers.SGD(0.05, momentum=0.9),
}
LOSSES = {
    "cce":        lambda: keras.losses.CategoricalCrossentropy(),
    "cce_smooth": lambda: keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    "kld":        lambda: keras.losses.KLDivergence(),
}

def build_model(rnn, cfg):
    Cell = layers.LSTM if rnn == "LSTM" else layers.GRU
    inp = keras.Input(shape=(MAXLEN,), dtype="int32")
    x = layers.Embedding(VOCAB, cfg["emb_dim"], mask_zero=True)(inp)
    x = layers.SpatialDropout1D(cfg["dropout"])(x)
    rnn_layer = Cell(cfg["units"])
    if cfg["bidirectional"]:
        rnn_layer = layers.Bidirectional(rnn_layer)
    x = rnn_layer(x)
    x = layers.Dropout(cfg["dropout"])(x)
    x = layers.Dense(cfg["units"], activation=cfg["activation"])(x)
    x = layers.Dropout(cfg["dropout"])(x)
    out = layers.Dense(N_CLASSES, activation="softmax")(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=OPTIMIZERS[cfg["optimizer"]](),
                  loss=LOSSES[cfg["loss"]]())
    return model

def train_model(rnn, cfg, verbose=0):
    keras.utils.set_random_seed(SEED)
    model = build_model(rnn, cfg)
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=4,
                                       restore_best_weights=True)
    hist = model.fit(X_tr, Y_tr, validation_data=(X_val, Y_val),
                     epochs=EPOCHS, batch_size=BATCH, callbacks=[es],
                     class_weight=CLASS_WEIGHT, verbose=verbose)
    return model, hist

In [ ]:
VAL_CSV = "../results/task3_val.csv"
cache = {}

def run_experiment(rnn, cfg):
    key = (rnn, json.dumps(cfg, sort_keys=True))
    if key in cache:
        return cache[key]
    model, hist = train_model(rnn, cfg)
    pred = model.predict(X_val, verbose=0).argmax(1)
    row = {
        "model": rnn, "preprocessing": PREPROC, **cfg,
        "epochs": len(hist.history["loss"]),
        "val_f1_macro": f1_score(y_val, pred, average="macro"),
        "val_f1_weighted": f1_score(y_val, pred, average="weighted"),
        "val_accuracy": accuracy_score(y_val, pred),
    }
    pd.DataFrame([row]).to_csv(VAL_CSV, mode="a", index=False,
                               header=not os.path.exists(VAL_CSV))
    keras.backend.clear_session()
    cache[key] = row
    print(f"{rnn} | {json.dumps(cfg)} -> f1_macro={row['val_f1_macro']:.4f}")
    return row

In [ ]:
BASE = dict(bidirectional=True, emb_dim=128, units=64, dropout=0.3,
            activation="relu", optimizer="adam", loss="cce")

GRID = {
    "bidirectional": [False, True],
    "emb_dim":    [64, 128, 200, 300],
    "units":      [32, 64, 128, 256],
    "dropout":    [0.0, 0.2, 0.3, 0.5],
    "optimizer":  list(OPTIMIZERS),
    "activation": ["relu", "tanh", "elu", "selu"],
    "loss":       list(LOSSES),
}

best_cfg = {}
for rnn in ["LSTM", "GRU"]:
    cfg = dict(BASE)
    for param, values in GRID.items():
        scores = {v: run_experiment(rnn, {**cfg, param: v})["val_f1_macro"] for v in values}
        cfg[param] = max(scores, key=scores.get)
        print(f"==> {rnn}: лучший {param} = {cfg[param]}\n")
    best_cfg[rnn] = cfg

print(json.dumps(best_cfg, indent=2))

In [ ]:
# сводка по перебору: лучшие значения каждого параметра
val = pd.read_csv(VAL_CSV)
for p in GRID:
    print(f"\n--- {p} ---")
    print(val.groupby(["model", p])["val_f1_macro"].max().unstack(0).round(4))

In [ ]:
# Финал: лучшие LSTM и GRU, метрики на тесте считаем один раз
TEST_CSV = "../results/task3_test.csv"
final = {}
for rnn, cfg in best_cfg.items():
    model, hist = train_model(rnn, cfg)
    pred = model.predict(X_test, verbose=0).argmax(1)
    row = {"model": rnn, "preprocessing": PREPROC, **cfg,
           "f1_macro": f1_score(y_test, pred, average="macro"),
           "f1_weighted": f1_score(y_test, pred, average="weighted"),
           "accuracy": accuracy_score(y_test, pred)}
    pd.DataFrame([row]).to_csv(TEST_CSV, mode="a", index=False,
                               header=not os.path.exists(TEST_CSV))
    final[rnn] = (row, pred)
    print(f"\n===== {rnn} =====")
    print(classification_report(y_test, pred, target_names=le.classes_.astype(str)))

best_rnn = max(final, key=lambda k: final[k][0]["f1_macro"])
fig, ax = plt.subplots(figsize=(7, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test, final[best_rnn][1], display_labels=le.classes_.astype(str),
    xticks_rotation=45, ax=ax, colorbar=False)
ax.set_title(f"{best_rnn}: матрица ошибок на тесте")
plt.tight_layout()
plt.savefig("../results/task3_confusion.png", dpi=200)
plt.show()